In [23]:
# -*- coding: utf-8 -*-
"""
================================================================================
 BIST TÜM (XUTUM) — FAKTÖR BAZLI RİSK-AYARLI MOMENTUM TARAYICI (Weekly Rebalance)
================================================================================
 Strateji  : Cross-Sectional Risk-Adjusted Momentum + Trend Kalitesi (ADX & R²)
 Rejim     : Fiyat > EMA50 > EMA200 (zorunlu boğa rejimi filtresi)
 Likidite  : 20 günlük ortalama işlem hacmi (TL) duvarı (zorunlu)
 Ceza      : RSI > 85 ve "Kauçuk Bant" (EMA21 / Bollinger üst bandından aşırı kopma)
 Çıktı     : Eşit ağırlıklı portföy için TOP 5 hisse + kısa Quant gerekçesi
 Kullanım  : pip install yfinance pandas numpy
             python bist_momentum_quant.py
 Not       : Haftada bir (tercihen Cuma kapanışı sonrası / hafta sonu) çalıştırın.
================================================================================
"""

import sys
import math
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings("ignore")
pd.set_option("mode.chained_assignment", None)

# ==============================================================================
# 1) PARAMETRELER
# ==============================================================================
MIN_BARS            = 210            # Minimum veri (>= EMA200 + tampon)
DOWNLOAD_PERIOD     = "420d"         # ~300 işlem barı için takvim günü
MIN_AVG_TL_VOLUME   = 50_000_000     # Likidite duvarı: 20g ort. hacim (TL)
MIN_PRICE           = 2.0            # Kuruşluk tahtaları ele
MAX_WORKERS         = 16             # ThreadPoolExecutor paralel indirme
TOP_N               = 5              # Portföye girecek hisse sayısı

MOM_WINDOW_LONG     = 126            # 6 aylık momentum penceresi
MOM_WINDOW_SHORT    = 63             # 3 aylık momentum penceresi
REG_WINDOW          = 90             # R² için doğrusal regresyon penceresi
ADX_PERIOD          = 14
RSI_PERIOD          = 14
ATR_PERIOD          = 14
ADX_MIN_TREND       = 25             # Trend kalitesi eşiği (bilgi amaçlı raporlanır)

RSI_OVERBOUGHT      = 85             # Mean-reversion ceza eşiği
STRETCH_ATR_LIMIT   = 3.0            # (Fiyat - EMA21)/ATR bu değeri aşarsa ceza
BB_PERIOD, BB_STD   = 20, 2.0

# ==============================================================================
# 2) BIST TÜM EVRENİ (statik liste)
# ==============================================================================
BIST_TICKERS = [
    "A1CAP","ADEL","AEFES","AGESA","AGHOL","AHGAZ","AKBNK","AKCNS","AKFGY","AKFYE",
    "AKGRT","AKSA","AKSEN","AKSGY","ALARK","ALBRK","ALCAR","ALCTL","ALFAS","ALKIM",
    "ANHYT","ANSGR","ARCLK","ARDYZ","ARENA","ARSAN","ASELS","ASTOR","ASUZU","ATAKP",
    "ATATP","AVGYO","AYDEM","AYEN","AYGAZ","BAGFS","BANVT","BARMA","BASGZ","BERA",
    "BFREN","BIENY","BIGCH","BIMAS","BINHO","BIOEN","BJKAS","BLCYT","BMSTL","BOBET",
    "BOSSA","BRISA","BRKSN","BRSAN","BRYAT","BTCIM","BUCIM","BURCE","BURVA","CANTE",
    "CCOLA","CEMAS","CEMTS","CEOEM","CIMSA","CLEBI","CRDFA","CRFSA","CVKMD","CWENE",
    "DAGI","DAPGM","DARDL","DENGE","DERIM","DESA","DESPC","DEVA","DGATE","DGNMO",
    "DITAS","DMSAS","DOAS","DOBUR","DOGUB","DOHOL","DOKTA","DYOBY","ECILC","ECZYT",
    "EGEEN","EGGUB","EGPRO","EGSER","EKGYO","ELITE","EMKEL","ENERY","ENJSA","ENKAI",
    "ERBOS","EREGL","ERSU","ESCOM","ESEN","ETILR","EUPWR","EUREN","FENER","FMIZP",
    "FONET","FORMT","FROTO","GARAN","GEDIK","GEDZA","GENIL","GENTS","GEREL","GESAN",
    "GLBMD","GLYHO","GOLTS","GOODY","GOZDE","GRNYO","GSDHO","GSRAY","GUBRF","GWIND",
    "HALKB","HATEK","HEKTS","HTTBT","HUBVC","HURGZ","IEYHO","IHEVA","IHGZT","IHLAS",
    "IHYAY","IMASM","INDES","INFO","INTEM","IPEKE","ISBIR","ISCTR","ISDMR","ISGYO",
    "ISKPL","ISMEN","IZENR","IZMDC","JANTS","KAPLM","KAREL","KARSN","KARTN","KATMR",
    "KAYSE","KCAER","KCHOL","KENT","KERVN","KLGYO","KLMSN","KLRHO","KMPUR","KNFRT",
    "KONTR","KONYA","KORDS","KOZAA","KOZAL","KRDMD","KRGYO","KRONT","KRSTL","KRTEK",
    "KUTPO","KZBGY","LIDER","LINK","LKMNH","LOGO","LUKSK","MAKIM","MAKTK","MANAS",
    "MARKA","MAVI","MEDTR","MEGAP","MERCN","MERKO","METRO","METUR","MGROS","MIATK",
    "MNDRS","MOBTL","MPARK","MRGYO","MRSHL","NATEN","NETAS","NIBAS","NTHOL","NUGYO",
    "NUHCM","OBASE","ODAS","ORCAY","ORGE","OSMEN","OSTIM","OTKAR","OYAKC","OYLUM",
    "OZKGY","PAGYO","PAPIL","PARSN","PASEU","PCILT","PENGD","PENTA","PETKM","PETUN",
    "PGSUS","PINSU","PKART","PKENT","PNSUT","POLHO","POLTK","PRKAB","PRKME","PRZMA",
    "QNBFB","QUAGR","RAYSG","REEDR","RODRG","RTALB","SAFKR","SAHOL","SAMAT","SANEL",
    "SANFM","SANKO","SARKY","SASA","SAYAS","SDTTR","SEKFK","SEKUR","SELEC","SELGD",
    "SEYKM","SILVR","SISE","SKBNK","SMRTG","SNGYO","SNKRN","SODSN","SOKM","SONME",
    "SUMAS","SUWEN","TABGD","TATGD","TAVHL","TCELL","TDGYO","TEZOL","TGSAS","THYAO",
    "TKFEN","TKNSA","TLMAN","TMSN","TOASO","TRCAS","TRGYO","TSGYO","TSKB","TSPOR",
    "TTKOM","TTRAK","TUCLK","TUKAS","TUPRS","TURSG","UFUK","ULAS","ULKER","ULUSE",
    "ULUUN","UNLU","USAK","VAKBN","VAKFN","VAKKO","VBTYZ","VERTU","VERUS","VESBE",
    "VESTL","VKGYO","YAPRK","YATAS","YEOTK","YESIL","YGGYO","YGYO","YKBNK","YKSLN",
    "YUNSA","YYLGD","ZEDUR","ZOREN","ZRGYO",
]

# ==============================================================================
# 3) TEKNİK GÖSTERGELER (saf pandas/numpy — harici TA kütüphanesi gerekmez)
# ==============================================================================
def ema(series: pd.Series, span: int) -> pd.Series:
    return series.ewm(span=span, adjust=False).mean()

def rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    h, l, c = df["High"], df["Low"], df["Close"]
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1/period, adjust=False).mean()

def adx(df: pd.DataFrame, period: int = 14) -> pd.Series:
    h, l, c = df["High"], df["Low"], df["Close"]
    up, dn = h.diff(), -l.diff()
    plus_dm  = np.where((up > dn) & (up > 0), up, 0.0)
    minus_dm = np.where((dn > up) & (dn > 0), dn, 0.0)
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr_w = tr.ewm(alpha=1/period, adjust=False).mean()
    plus_di  = 100 * pd.Series(plus_dm,  index=df.index).ewm(alpha=1/period, adjust=False).mean() / atr_w
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/period, adjust=False).mean() / atr_w
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    return dx.ewm(alpha=1/period, adjust=False).mean()

def linreg_r2_slope(log_close: np.ndarray):
    """Log-fiyat üzerinde doğrusal regresyon: (R², yıllıklandırılmış eğim %)."""
    n = len(log_close)
    x = np.arange(n, dtype=float)
    x_mean, y_mean = x.mean(), log_close.mean()
    cov  = ((x - x_mean) * (log_close - y_mean)).sum()
    varx = ((x - x_mean) ** 2).sum()
    if varx == 0:
        return 0.0, 0.0
    slope = cov / varx
    y_hat = y_mean + slope * (x - x_mean)
    ss_res = ((log_close - y_hat) ** 2).sum()
    ss_tot = ((log_close - y_mean) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    ann_slope_pct = (math.exp(slope * 252) - 1) * 100
    return max(r2, 0.0), ann_slope_pct

# ==============================================================================
# 4) VERİ İNDİRME (ThreadPoolExecutor)
# ==============================================================================
def fetch_one(ticker: str):
    try:
        df = yf.download(f"{ticker}.IS", period=DOWNLOAD_PERIOD, interval="1d",
                         auto_adjust=True, progress=False, threads=False)
        if df is None or df.empty:
            return ticker, None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.dropna(subset=["Close", "High", "Low", "Volume"])
        return ticker, (df if len(df) >= MIN_BARS else None)
    except Exception:
        return ticker, None

def fetch_all(tickers):
    data = {}
    print(f"[i] {len(tickers)} hisse için veri indiriliyor ({MAX_WORKERS} paralel iş parçacığı)...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(fetch_one, t): t for t in tickers}
        done = 0
        for fut in as_completed(futures):
            t, df = fut.result()
            done += 1
            if df is not None:
                data[t] = df
            if done % 50 == 0:
                print(f"    ... {done}/{len(tickers)} tamamlandı")
    print(f"[i] Yeterli geçmişe (≥{MIN_BARS} bar) sahip hisse: {len(data)}")
    return data

# ==============================================================================
# 5) HİSSE BAZLI METRİK HESABI
# ==============================================================================
def compute_metrics(ticker: str, df: pd.DataFrame):
    close = df["Close"]
    price = float(close.iloc[-1])
    if price < MIN_PRICE:
        return None

    # --- Likidite duvarı (zorunlu) -------------------------------------------
    tl_vol = (df["Close"] * df["Volume"])
    avg_tl_vol_20 = float(tl_vol.tail(20).mean())
    if avg_tl_vol_20 < MIN_AVG_TL_VOLUME:
        return None

    # --- Temel trend rejimi: Fiyat > EMA50 > EMA200 (zorunlu) ----------------
    ema21_s, ema50_s, ema200_s = ema(close, 21), ema(close, 50), ema(close, 200)
    ema21, ema50, ema200 = float(ema21_s.iloc[-1]), float(ema50_s.iloc[-1]), float(ema200_s.iloc[-1])
    if not (price > ema50 > ema200):
        return None
    ema50_slope = float(ema50_s.iloc[-1] / ema50_s.iloc[-10] - 1) * 100  # 10 barlık eğim %

    # --- Risk-Adjusted Momentum (Sharpe benzeri, yıllıklandırılmış) ----------
    rets = close.pct_change().dropna()
    def risk_adj(window):
        if len(close) < window + 1:
            return np.nan, np.nan
        tot_ret = float(close.iloc[-1] / close.iloc[-window - 1] - 1)
        vol = float(rets.tail(window).std())
        if vol <= 0:
            return np.nan, tot_ret
        ram = (tot_ret / window) / vol * math.sqrt(252)   # günlük ort. getiri / vol
        return ram, tot_ret

    ram_126, ret_126 = risk_adj(MOM_WINDOW_LONG)
    ram_63,  ret_63  = risk_adj(MOM_WINDOW_SHORT)
    if np.isnan(ram_126) or np.isnan(ram_63):
        return None
    ram_blend = 0.6 * ram_126 + 0.4 * ram_63

    # --- Trend kalitesi: ADX + Regresyon R² ----------------------------------
    adx_val = float(adx(df, ADX_PERIOD).iloc[-1])
    r2, reg_slope_ann = linreg_r2_slope(np.log(close.tail(REG_WINDOW).values))
    if reg_slope_ann <= 0:          # eğimi negatif olan "pürüzsüz düşüş" elenir
        return None

    # --- Hacim ivmesi ----------------------------------------------------------
    vol_surge = float(tl_vol.tail(20).mean() / max(tl_vol.tail(60).mean(), 1.0))

    # --- Aşırı ısınma metrikleri (ceza tarafı) --------------------------------
    rsi_val = float(rsi(close, RSI_PERIOD).iloc[-1])
    atr_val = float(atr(df, ATR_PERIOD).iloc[-1])
    stretch_atr = (price - ema21) / atr_val if atr_val > 0 else 0.0
    bb_mid = close.rolling(BB_PERIOD).mean()
    bb_up  = float((bb_mid + BB_STD * close.rolling(BB_PERIOD).std()).iloc[-1])
    bb_breach_pct = max(0.0, (price / bb_up - 1) * 100) if bb_up > 0 else 0.0

    return dict(ticker=ticker, price=price, avg_tl_vol_20=avg_tl_vol_20,
                ram_blend=ram_blend, ram_126=ram_126, ram_63=ram_63,
                ret_126=ret_126 * 100, ret_63=ret_63 * 100,
                adx=adx_val, r2=r2, reg_slope_ann=reg_slope_ann,
                vol_surge=vol_surge, ema50_slope=ema50_slope,
                rsi=rsi_val, stretch_atr=stretch_atr, bb_breach_pct=bb_breach_pct)

# ==============================================================================
# 6) CROSS-SECTIONAL PUANLAMA (0-100) + CEZALAR
# ==============================================================================
def score_universe(rows):
    df = pd.DataFrame(rows).set_index("ticker")
    pct = lambda col: df[col].rank(pct=True)   # evren içi yüzdelik sıra (0-1)

    base = (
        40 * pct("ram_blend")   +   # Risk-ayarlı momentum (ana faktör)
        15 * pct("adx")         +   # Trend gücü
        15 * pct("r2")          +   # Trend pürüzsüzlüğü
        10 * pct("ret_63")      +   # Saf kısa vadeli ivme
        10 * pct("vol_surge")   +   # Hacim ivmesi / kurumsal ilgi
        10 * pct("ema50_slope")     # Rejim sağlığı (EMA50 eğimi)
    )

    # --- Mean-reversion (kauçuk bant) cezaları --------------------------------
    pen_rsi = np.where(df["rsi"] > RSI_OVERBOUGHT, 15.0, 0.0)
    pen_str = np.clip((df["stretch_atr"] - STRETCH_ATR_LIMIT) * 8.0, 0, 20)
    pen_bb  = np.where(df["bb_breach_pct"] > 2.0, 10.0, 0.0)

    df["penalty"] = pen_rsi + pen_str + pen_bb
    df["score"]   = (base - df["penalty"]).clip(0, 100).round(1)
    return df.sort_values("score", ascending=False)

# ==============================================================================
# 7) RAPORLAMA
# ==============================================================================
def build_commentary(t, r):
    notes = []
    notes.append(f"6A risk-ayarlı momentum {r.ram_126:.2f} (getiri %{r.ret_126:.0f}, "
                 f"düşük dalgalanmayla kazandıran profil)")
    q = "güçlü ve pürüzsüz" if (r.adx >= ADX_MIN_TREND and r.r2 >= 0.75) else \
        "sağlıklı" if r.adx >= ADX_MIN_TREND else "gelişmekte olan"
    notes.append(f"trend {q}: ADX {r.adx:.0f}, R² {r.r2:.2f}, "
                 f"regresyon eğimi yıllık ~%{r.reg_slope_ann:.0f}")
    notes.append(f"Fiyat>EMA50>EMA200 rejimi teyitli; 20g ort. hacim "
                 f"{r.avg_tl_vol_20/1e6:.0f}M TL, hacim ivmesi x{r.vol_surge:.2f}")
    if r.penalty > 0:
        notes.append(f"uyarı: aşırı ısınma cezası -{r.penalty:.0f}p "
                     f"(RSI {r.rsi:.0f}, EMA21'den {r.stretch_atr:.1f} ATR uzakta)")
    else:
        notes.append(f"aşırılık yok (RSI {r.rsi:.0f}, EMA21'e {r.stretch_atr:.1f} ATR mesafe) "
                     f"— kauçuk bant riski düşük")
    return " | ".join(notes)

def main():
    data = fetch_all(BIST_TICKERS)
    rows = []
    for t, df in data.items():
        try:
            m = compute_metrics(t, df)
            if m:
                rows.append(m)
        except Exception:
            continue

    if len(rows) < TOP_N:
        print("[!] Filtreleri geçen yeterli hisse yok. Likidite/rejim koşulları çok sıkı olabilir "
              "veya piyasa genel ayı rejiminde (nakitte kalmak da bir pozisyondur).")
        sys.exit(0)

    ranked = score_universe(rows)
    top = ranked.head(TOP_N)

    print("\n" + "=" * 78)
    print(f"  HAFTALIK REBALANCE — TOP {TOP_N} RİSK-AYARLI MOMENTUM (Eşit Ağırlık: her biri %{100//TOP_N})")
    print(f"  Filtreyi geçen evren: {len(ranked)} hisse | Tarih: {pd.Timestamp.today():%d.%m.%Y}")
    print("=" * 78)
    for i, (t, r) in enumerate(top.iterrows(), 1):
        print(f"\n  #{i}  {t:<6}  Fiyat: {r.price:>8.2f} TL   SKOR: {r.score:>5.1f}/100")
        print(f"      {build_commentary(t, r)}")
    print("\n" + "-" * 78)
    print("  Not: Bu çıktı bir analiz aracıdır, yatırım tavsiyesi değildir.")
    print("-" * 78)

if __name__ == "__main__":
    main()

[i] 325 hisse için veri indiriliyor (16 paralel iş parçacığı)...
    ... 50/325 tamamlandı


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:
1 Failed download:


    ... 100/325 tamamlandı
    ... 150/325 tamamlandı


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['IPEKE.IS']: YFPricesMissingError('possibly delisted; no price data found  (period=420d) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['IPEKE.IS']: YFPricesMissingError('possibly delisted; no price data found  (period=420d) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['KOZAL.IS', 'KOZAA.IS']: YFPricesMissingError('possibly delisted; no price data found  (period=420d) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['KOZAL.IS', 'KOZAA.IS']: YFPricesMissingError('possibly delisted; no price data found  (period=420d) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:


    ... 200/325 tamamlandı


...

    ... 250/325 tamamlandı


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['QNBFB.IS']: YFPricesMissingError('possibly delisted; no price data found  (period=420d) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:
1 Failed download:


    ... 300/325 tamamlandı


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['YGYO.IS']: YFPricesMissingError('possibly delisted; no price data found  (period=420d) (Yahoo error = "No data found, symbol may be delisted")')


[i] Yeterli geçmişe (≥210 bar) sahip hisse: 301

  HAFTALIK REBALANCE — TOP 5 RİSK-AYARLI MOMENTUM (Eşit Ağırlık: her biri %20)
  Filtreyi geçen evren: 19 hisse | Tarih: 10.07.2026

  #1  AHGAZ   Fiyat:    37.08 TL   SKOR:  88.9/100
      6A risk-ayarlı momentum 2.97 (getiri %66, düşük dalgalanmayla kazandıran profil) | trend gelişmekte olan: ADX 25, R² 0.89, regresyon eğimi yıllık ~%437 | Fiyat>EMA50>EMA200 rejimi teyitli; 20g ort. hacim 256M TL, hacim ivmesi x1.28 | aşırılık yok (RSI 60, EMA21'e 1.0 ATR mesafe) — kauçuk bant riski düşük

  #2  DAGI    Fiyat:     9.66 TL   SKOR:  87.9/100
      6A risk-ayarlı momentum 2.03 (getiri %51, düşük dalgalanmayla kazandıran profil) | trend güçlü ve pürüzsüz: ADX 47, R² 0.86, regresyon eğimi yıllık ~%319 | Fiyat>EMA50>EMA200 rejimi teyitli; 20g ort. hacim 127M TL, hacim ivmesi x1.54 | aşırılık yok (RSI 63, EMA21'e 1.4 ATR mesafe) — kauçuk bant riski düşük

  #3  BARMA   Fiyat:    69.70 TL   SKOR:  80.5/100
      6A risk-ayarlı momentum 3.55 (g